# Animated boundary mean spectra (mean-cov + block-vs-residual)

Part of the experiments_anim split (1: boundary mean spectra, 2: eigval spectra, 3: coupling heatmaps, 4: layerwise RankMe) — split so saved outputs stay small enough for the editor.


In [ ]:
import os, sys
import warnings
import importlib
if os.path.basename(os.getcwd()) == 'analysis':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import utils.model_registry, utils.accessor
importlib.reload(utils.model_registry)   # deps first: reload(_lib) alone re-imports cached modules
importlib.reload(utils.accessor)
from analysis import experiments_lib as _lib
importlib.reload(_lib)
from analysis.experiments_lib import (
    build_hooks, get_ys, get_series_y, panel_palettes, smooth_spectrum,
    submatrix, block_mean_cos, model_name_options, YVAR_LABELS, XVAR_FNS)
# nanochat zero-inits c_proj: step-0 attn/mlp.out frames are all-zero; ylim is pinned
# globally so the log-autoscale warning on those frames is pure noise.
warnings.filterwarnings('ignore', message='Data has no positive values')

In [ ]:
# Config this notebook needs (kept out of the generic lib):
BLOCK_REPR = 'block_representations_all'         # all-layer cov runs (the 4 main models)
BLOCK_SAMPLES = 'block_representations_samples'  # samples runs (also carry 160m/410m)
SRC = {                                          # model -> source for spectra/means/eigvals
    'pythia-160m-deduped':  BLOCK_SAMPLES,
    'pythia-410m-deduped':  BLOCK_SAMPLES,
    'pythia-1b-deduped':    BLOCK_REPR,
    'pythia-6.9b-deduped':  BLOCK_REPR,
    'OLMo-2-0425-1B':       BLOCK_REPR,
    'OLMo-2-1124-7B':       BLOCK_REPR,
    'nanochat-d12':         'nanochat_samples',
}
SAMPLES_SRC = {m: 'nanochat_samples' if m == 'nanochat-d12' else BLOCK_SAMPLES for m in SRC}
measured_br = {m: list(range(L)) for m, L in [
    ('pythia-160m-deduped', 12), ('pythia-410m-deduped', 24),
    ('pythia-1b-deduped', 16),   ('pythia-6.9b-deduped', 32),
    ('OLMo-2-0425-1B', 16),      ('OLMo-2-1124-7B', 32),
    ('nanochat-d12', 12)]}
# sequential blocks (OLMo-2, nanochat) expose a distinct mlp.in; parallel Pythia aliases attn.in
has_mlp_in = lambda model: 'olmo' in model.lower() or 'nanochat' in model.lower()
HK = build_hooks()                     # hook-name -> (leaf, metric) table
def bnd(model, prefix, ms):
    return [(SRC[model], HK[f'{prefix}{l}_{m.upper()[:2]}'], f'{m} {l} {prefix}')
            for l in measured_br[model] for m in ms]
attn_in  = lambda model, ms=['Au']: bnd(model, 'AI', ms)
attn_out = lambda model, ms=['Au']: bnd(model, 'AO', ms)
mlp_in   = lambda model, ms=['Au']: bnd(model, 'MI', ms)
mlp_out  = lambda model, ms=['Au']: bnd(model, 'MO', ms)

In [ ]:
# Animated-spectra engine (analysis/spectrum_anim.py): inject this notebook's data
# backend, expose animate_spectra. The notebook drives animations via animate_spectra /
# anim_meancov / anim_blkres (no plot_spectrum needed here).
import analysis.spectrum_anim as sa
importlib.reload(sa)   # pick up engine edits without restarting the kernel
sa.configure(get_series_y=get_series_y, get_ys=get_ys,
             panel_palettes=panel_palettes, smooth_spectrum=smooth_spectrum,
             submatrix=submatrix, block_mean_cos=block_mean_cos,
             model_name_options=model_name_options, YVAR_LABELS=YVAR_LABELS, XVAR_FNS=XVAR_FNS)
animate_spectra = sa.animate_spectra

## Animated spectra — all boundaries, all models
Profile spectra ($p_j=|\langle\hat\mu, v_j\rangle|^2$) and energy-weighted profiles, animated across checkpoints. Two sections mirroring the source notebook:
- **Mean-covariance** — μ inside the *centered* covariance eigenbasis (`acts_mean_metrics`).
- **Block-vs-residual** — the block-output mean vs the input residual it joins (`mean_metrics_blk_vs_res`, at the sub-block node).

One animation per boundary (profile + weighted, labelled). Run a model's cell to view inline; each animation is ~7 MB. Pass `save_dir='analysis/figures/spectrum_anim'` to `anim_meancov`/`anim_blkres` to render mp4s (fire-and-forget) instead of inline. Smoothing is on by default (`smooth=22, peak=3` — shared `smooth_spectrum`: Savitzky-Golay with an upper-envelope bias that keeps peak heights and edges); pass `smooth=0` to disable.

In [ ]:
from IPython.display import display

_MC_BNDS = [('MLP out', mlp_out), ('Attn out', attn_out), ('MLP in', mlp_in), ('Attn in', attn_in)]
_BR_BNDS = [('MLP out', mlp_out), ('Attn out', attn_out)]
# final-norm "boundary": two lines (before/after final norm) instead of layers; labels keep
# the `<word> <legend>` shape so `_u_meancov`'s lbl.split()[1] yields 'bfn'/'afn'
final_norm = lambda model, ms=('Au',): [(SRC[model], (leaf, 'acts_mean_metrics'), f'norm {lbl}')
                                        for leaf, lbl in [('before_final_norm', 'bfn'),
                                                          ('after_final_norm', 'afn')]]
_FN_BNDS = [('Final norm', final_norm)]

# natural y-scale per metric (matches the source notebook); override via (name, {'ylog': ...})
_METRIC_YLOG = {'mean_norm': True, 'mean_frac': True, 'rayleigh': True, 'mahalanobis': True,
                'pr': True, 'pr_weighted': True, 'rayleigh_normed': False, 'max_overlap': False,
                'top_overlap': False, 'max_overlap_idx': False, 'centroid_idx': False,
                'centroid_idx_weighted': False}

def _u_meancov(model, bnd):                         # legend label = layer number only
    return [(s, (h[0], 'acts_mean_metrics'), lbl.split()[1]) for s, h, lbl in bnd(model, ['Au'])]

def _u_blkres(model, bnd):
    return [(s, (h[0].rsplit('.', 1)[0], 'mean_metrics_blk_vs_res'), lbl.split()[1])
            for s, h, lbl in bnd(model, ['Au'])]

def _strip(m, u, model):                            # m = name or (name, {opts})
    name, mopts = m if isinstance(m, tuple) else (m, {})
    return (name, u, [model], {'kind': 'strip', 'ylog': _METRIC_YLOG.get(name, True), **mopts})

def _anim_section(model, bnds, u_fn, tag, save_dir=None, metrics=(), **kw):
    for name, bnd in bnds:
        u = u_fn(model, bnd)
        panels = [('profile', u, [model], {'title': f'{name} — profile $p_j$'}),
                  *[_strip(m, u, model) for m in metrics],          # thin band between the two
                  ('profile_weighted', u, [model], {'title': f'{name} — weighted'})]
        # smooth: ONLY the mean-to-eigvec overlap profiles are noisy (sorted spectra never are)
        opts = dict(xlog=False, ylog=True, smooth=22, peak=3, model=model,
                    suptitle=f'{tag} — {name} — {model}', **kw)
        safe = f'{tag}_{name}_{model}'.replace(' ', '_')             # save (if any) is a side effect:
        save = f'{save_dir}/{safe}.mp4' if save_dir else None        # always render inline
        display(animate_spectra(panels, save=save, **opts))

def anim_meancov(model, save_dir=None, metrics=(), bnds=None, **kw):
    if bnds is None:
        bnds = [b for b in _MC_BNDS if b[0] != 'MLP in' or has_mlp_in(model)]
    _anim_section(model, bnds, _u_meancov, 'Mean-cov', save_dir, metrics=metrics, **kw)

def anim_blkres(model, save_dir=None, metrics=(), **kw):
    _anim_section(model, _BR_BNDS, _u_blkres, 'Blk-vs-res', save_dir, metrics=metrics, **kw)

### pythia-160m-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-160m-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-160m-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### pythia-410m-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-410m-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-410m-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### pythia-1b-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-1b-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-1b-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### pythia-6.9b-deduped

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('pythia-6.9b-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('pythia-6.9b-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### OLMo-2-0425-1B

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('OLMo-2-0425-1B', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('OLMo-2-0425-1B', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### OLMo-2-1124-7B

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('OLMo-2-1124-7B', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('OLMo-2-1124-7B', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

### nanochat-d12

In [ ]:
# Mean-covariance spectra, per boundary (profile + weighted)
anim_meancov('nanochat-d12', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

In [ ]:
# Block-output mean vs the residual it joins, per boundary
anim_blkres('nanochat-d12', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'])

## Final norm — before vs after
Same panels as Mean-cov (profile + 4 metric strips + weighted), but the sources are the two final-norm leaves (`before_final_norm` = bfn, `after_final_norm` = afn) instead of layers.

In [ ]:
anim_meancov('pythia-160m-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], bnds=_FN_BNDS)

In [ ]:
anim_meancov('pythia-410m-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], bnds=_FN_BNDS)

In [ ]:
anim_meancov('pythia-1b-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], bnds=_FN_BNDS)

In [ ]:
anim_meancov('pythia-6.9b-deduped', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], bnds=_FN_BNDS)

In [ ]:
anim_meancov('OLMo-2-0425-1B', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], bnds=_FN_BNDS)

In [ ]:
anim_meancov('OLMo-2-1124-7B', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], bnds=_FN_BNDS)

In [ ]:
anim_meancov('nanochat-d12', save_dir='analysis/figures/animations', metrics=['mean_norm', 'rayleigh_normed', 'mahalanobis', 'centroid_idx_weighted'], bnds=_FN_BNDS)